# BÀI 4: Neural Network phân lớp bộ dữ liệu Iris bằng K-Fold Cross Validation (K = 10).


## Chuẩn bị thư viện

In [1]:
print("NguyenDucTinh - 6451071077")

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import os
import joblib

NguyenDucTinh - 6451071077


## Câu a: Đọc bộ dữ liệu Iris từ sklearn

In [2]:
print("NguyenDucTinh - 6451071077")

# Đọc bộ dữ liệu Iris
iris = load_iris()

# Lấy dữ liệu đầu vào
data = iris.data

# Lấy nhãn
target = iris.target

# Tên các lớp
class_names = iris.target_names

print("Dữ liệu Iris đã được tải thành công!")

print("Kích thước data:", data.shape)
print("Kích thước target:", target.shape)

print("Tên các lớp:", class_names)

print("\n5 mẫu dữ liệu đầu tiên:")
print(data[:5])

print("\n5 nhãn đầu tiên:")
print(target[:5])

NguyenDucTinh - 6451071077
Dữ liệu Iris đã được tải thành công!
Kích thước data: (150, 4)
Kích thước target: (150,)
Tên các lớp: ['setosa' 'versicolor' 'virginica']

5 mẫu dữ liệu đầu tiên:
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]

5 nhãn đầu tiên:
[0 0 0 0 0]


## Câu b: Chuẩn hóa dữ liệu

In [3]:
print("NguyenDucTinh - 6451071077")

# Khởi tạo bộ chuẩn hóa
scaler = MinMaxScaler()

# Chuẩn hóa dữ liệu
data = scaler.fit_transform(data)

print("Dữ liệu sau khi chuẩn hóa:")
print(data[:5])

print("\nGiá trị nhỏ nhất:", data.min())
print("Giá trị lớn nhất:", data.max())

print("\nĐã chuẩn hóa dữ liệu thành công!")

NguyenDucTinh - 6451071077
Dữ liệu sau khi chuẩn hóa:
[[0.22222222 0.625      0.06779661 0.04166667]
 [0.16666667 0.41666667 0.06779661 0.04166667]
 [0.11111111 0.5        0.05084746 0.04166667]
 [0.08333333 0.45833333 0.08474576 0.04166667]
 [0.19444444 0.66666667 0.06779661 0.04166667]]

Giá trị nhỏ nhất: 0.0
Giá trị lớn nhất: 1.0

Đã chuẩn hóa dữ liệu thành công!


## Câu c: Định nghĩa hàm create_model

In [4]:
print("NguyenDucTinh - 6451071077")

def create_model():

    # Khởi tạo Neural Network
    model = Sequential()

    # Input layer và Hidden layer 1
    model.add(Dense(
        10,
        input_shape=(4,),
        activation='relu'
    ))

    # Hidden layer 2
    model.add(Dense(
        20,
        activation='relu'
    ))

    # Output layer gồm 3 lớp
    model.add(Dense(
        3,
        activation='softmax'
    ))

    # Biên dịch mô hình
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Kiểm tra mô hình
model = create_model()

model.summary()

NguyenDucTinh - 6451071077


c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

## Câu d: Huấn luyện K-Fold Cross Validation

In [5]:
print("NguyenDucTinh - 6451071077")

# Khởi tạo KFold với 10 Fold
kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# Các mảng lưu kết quả từng Fold
models = []

accuracy_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []

# Lưu scaler tương ứng với từng Fold
scalers = []

# Tạo thư mục lưu mô hình
os.makedirs("iris_kfold_models", exist_ok=True)

print("Đã khởi tạo KFold thành công!")

NguyenDucTinh - 6451071077
Đã khởi tạo KFold thành công!


In [6]:
print("NguyenDucTinh - 6451071077")

fold_no = 1

for train_index, val_index in kfold.split(data):

    print("\n====================================")
    print("ĐANG HUẤN LUYỆN FOLD:", fold_no)
    print("====================================")

    # Chia dữ liệu train và validation
    X_train_fold = data[train_index]
    X_val_fold = data[val_index]

    y_train_fold = target[train_index]
    y_val_fold = target[val_index]

    # Tạo mô hình mới
    model = create_model()

    # Huấn luyện mô hình
    model.fit(
        X_train_fold,
        y_train_fold,
        epochs=100,
        batch_size=1,
        verbose=0
    )

    # Dự đoán tập validation
    y_pred_prob = model.predict(
        X_val_fold,
        verbose=0
    )

    y_pred = np.argmax(y_pred_prob, axis=1)

    # Tính Accuracy
    accuracy = accuracy_score(
        y_val_fold,
        y_pred
    )

    # Tính Precision
    precision = precision_score(
        y_val_fold,
        y_pred,
        average='macro',
        zero_division=0
    )

    # Tính Recall
    recall = recall_score(
        y_val_fold,
        y_pred,
        average='macro',
        zero_division=0
    )

    # Tính F1-score
    f1 = f1_score(
        y_val_fold,
        y_pred,
        average='macro',
        zero_division=0
    )

    # Lưu mô hình và các kết quả
    models.append(model)

    accuracy_per_fold.append(accuracy)
    precision_per_fold.append(precision)
    recall_per_fold.append(recall)
    f1_per_fold.append(f1)

    # Lưu mô hình sau mỗi Fold
    model_path = f"iris_kfold_models/model_fold_{fold_no}.keras"

    model.save(model_path)

    print("\nKẾT QUẢ FOLD:", fold_no)

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    print("Đã lưu mô hình:", model_path)

    fold_no += 1

print("\nĐÃ HOÀN THÀNH 10 FOLD!")

NguyenDucTinh - 6451071077

ĐANG HUẤN LUYỆN FOLD: 1

KẾT QUẢ FOLD: 1
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
Đã lưu mô hình: iris_kfold_models/model_fold_1.keras

ĐANG HUẤN LUYỆN FOLD: 2


c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



KẾT QUẢ FOLD: 2
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
Đã lưu mô hình: iris_kfold_models/model_fold_2.keras

ĐANG HUẤN LUYỆN FOLD: 3


c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



KẾT QUẢ FOLD: 3
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
Đã lưu mô hình: iris_kfold_models/model_fold_3.keras

ĐANG HUẤN LUYỆN FOLD: 4


c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


... (9 outputs left out)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



KẾT QUẢ FOLD: 9
Accuracy: 0.9333333333333333
Precision: 0.9444444444444445
Recall: 0.9333333333333332
F1-score: 0.9326599326599326
Đã lưu mô hình: iris_kfold_models/model_fold_9.keras

ĐANG HUẤN LUYỆN FOLD: 10


c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



KẾT QUẢ FOLD: 10
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
Đã lưu mô hình: iris_kfold_models/model_fold_10.keras

ĐÃ HOÀN THÀNH 10 FOLD!


## Câu e: Tính Average Accuracy, Recall, Precision, F1-score

In [7]:
print("NguyenDucTinh - 6451071077")

# Tính trung bình kết quả của 10 Fold
average_accuracy = np.mean(accuracy_per_fold)

average_precision = np.mean(precision_per_fold)

average_recall = np.mean(recall_per_fold)

average_f1 = np.mean(f1_per_fold)

print("========== KẾT QUẢ TRUNG BÌNH 10 FOLD ==========")

print("Average Accuracy:", average_accuracy)

print("Average Precision:", average_precision)

print("Average Recall:", average_recall)

print("Average F1-score:", average_f1)

NguyenDucTinh - 6451071077
========== KẾT QUẢ TRUNG BÌNH 10 FOLD ==========
Average Accuracy: 0.9666666666666666
Average Precision: 0.9715873015873017
Average Recall: 0.9672222222222222
Average F1-score: 0.9646721796721798


## Câu f: Tính Average Precision, Recall, F1-score cho từng Class

In [8]:
print("NguyenDucTinh - 6451071077")

from sklearn.metrics import precision_recall_fscore_support

# Mảng lưu kết quả từng lớp
precision_per_class = []
recall_per_class = []
f1_per_class = []

# Chạy lại từng Fold để lấy chỉ số của từng Class
for fold_index, (train_index, val_index) in enumerate(kfold.split(data)):

    # Lấy dữ liệu validation
    X_val_fold = data[val_index]
    y_val_fold = target[val_index]

    # Lấy mô hình đã huấn luyện của Fold
    model = models[fold_index]

    # Dự đoán
    y_pred_prob = model.predict(X_val_fold, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # Tính Precision, Recall, F1-score cho từng lớp
    precision, recall, f1, support = precision_recall_fscore_support(
        y_val_fold,
        y_pred,
        labels=[0, 1, 2],
        zero_division=0
    )

    precision_per_class.append(precision)
    recall_per_class.append(recall)
    f1_per_class.append(f1)

# Tính trung bình theo từng Class
average_precision_class = np.mean(precision_per_class, axis=0)

average_recall_class = np.mean(recall_per_class, axis=0)

average_f1_class = np.mean(f1_per_class, axis=0)

# In kết quả
print("========== KẾT QUẢ TRUNG BÌNH TỪNG CLASS ==========")

for i in range(3):

    print("\nClass:", i, "-", class_names[i])

    print("Average Precision:", average_precision_class[i])

    print("Average Recall:", average_recall_class[i])

    print("Average F1-score:", average_f1_class[i])

NguyenDucTinh - 6451071077
========== KẾT QUẢ TRUNG BÌNH TỪNG CLASS ==========

Class: 0 - setosa
Average Precision: 1.0
Average Recall: 1.0
Average F1-score: 1.0

Class: 1 - versicolor
Average Precision: 0.9690476190476192
Average Recall: 0.9416666666666668
Average F1-score: 0.9489310689310688

Class: 2 - virginica
Average Precision: 0.9457142857142855
Average Recall: 0.96
Average F1-score: 0.9450854700854702


## Câu g: Chọn mô hình có F1-score tốt nhất

In [9]:
print("NguyenDucTinh - 6451071077")

# Tìm vị trí Fold có F1-score cao nhất
best_index = np.argmax(f1_per_fold)

# Lấy mô hình tốt nhất
best_model = models[best_index]

print("Fold có F1-score tốt nhất:", best_index + 1)

print("F1-score tốt nhất:", f1_per_fold[best_index])

print("Accuracy của mô hình:", accuracy_per_fold[best_index])

print("Đã chọn mô hình tốt nhất thành công!")


# Khai báo 3 mẫu dữ liệu mới
new_data = np.array([
    [6.2, 2.9, 4.3, 1.3],
    [5.1, 3.5, 1.4, 0.2],
    [7.3, 2.8, 6.4, 2.1]
])

# Chuẩn hóa dữ liệu mới bằng scaler đã sử dụng
new_data_scaled = scaler.transform(new_data)

# Sử dụng mô hình có F1-score tốt nhất để dự đoán
predictions = best_model.predict(new_data_scaled)

# Lấy nhãn dự đoán
predicted_classes = np.argmax(predictions, axis=1)

# In kết quả
print("========== KẾT QUẢ PHÂN LỚP 3 MẪU MỚI ==========")

for i in range(len(new_data)):

    print("\nMẫu dữ liệu:", i + 1)

    print("Dữ liệu:", new_data[i])

    print("Nhãn dự đoán:", predicted_classes[i])

    print("Tên lớp:", class_names[predicted_classes[i]])

    print("Xác suất dự đoán:", predictions[i][predicted_classes[i]])

NguyenDucTinh - 6451071077
Fold có F1-score tốt nhất: 1
F1-score tốt nhất: 1.0
Accuracy của mô hình: 1.0
Đã chọn mô hình tốt nhất thành công!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
========== KẾT QUẢ PHÂN LỚP 3 MẪU MỚI ==========

Mẫu dữ liệu: 1
Dữ liệu: [6.2 2.9 4.3 1.3]
Nhãn dự đoán: 1
Tên lớp: versicolor
Xác suất dự đoán: 0.99950945

Mẫu dữ liệu: 2
Dữ liệu: [5.1 3.5 1.4 0.2]
Nhãn dự đoán: 0
Tên lớp: setosa
Xác suất dự đoán: 0.999998

Mẫu dữ liệu: 3
Dữ liệu: [7.3 2.8 6.4 2.1]
Nhãn dự đoán: 2
Tên lớp: virginica
Xác suất dự đoán: 0.9994937
